# Study 881 — Jobless-Claims Sector Rotation — the teardown

The predictive Newey-West slope, the COVID-sensitivity / winsor / Spearman triple, the two-era cut, the permutation placebo, the costed rotation timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '1998-12-31', 'end': '2026-06-30', 'n_months': 331, 'n': 329, 'fingerprint': '43e0a9ec1a15', 'spread_mean_pct': 0.251, 'spread_sd_pct': 4.37, 'slope': 0.0177, 't_nw': 6.39, 'r2': 0.0127, 'corr': 0.1125, 'ex_covid_slope': 0.0623, 'ex_covid_t': 1.49, 'ex_covid_n': 318, 'winsor_slope': 0.0504, 'winsor_t': 1.43, 'spearman_rho': 0.0189, 'spearman_p': 0.733, 'era_early_slope': 0.1222, 'era_early_t': 1.41, 'era_early_n': 156, 'era_late_slope': 0.0164, 'era_late_t': 6.83, 'era_late_n': 173, 'placebo_obs': 0.0177, 'placebo_mean': -3e-05, 'placebo_sd': 0.0089, 'placebo_p': 0.053, 'timer0_gross': 0.89, 'timer0_net': 0.39, 'timer0_t': 0.14, 'timer10_gross': 0.89, 'timer10_net': -1.91, 'timer10_t': -0.66, 'n_switches': 168, 'null_mean_t': 0.16, 'null_sd_t': 0.95, 'null_fire': 1, 'planted_slope': -0.4651, 'planted_t': -13.85}

## The headline — predictive regression  `spread_{t+1} ~ dclaims_t`

Newey-West(6) regression of the forward **cyclical − defensive** spread on the 4-week claims change. The claim needs a **negative** slope.

In [2]:
print(f"slope        : {R['slope']:+.4f}   NW(6) t = {R['t_nw']:+.2f}   R2 = {R['r2']:.4f}")
print(f"correlation  : {R['corr']:+.4f}   (n={R['n']})")
print(f"spread mean  : {R['spread_mean_pct']:+.3f}%/mo (sd {R['spread_sd_pct']:.2f}%)")
print('sign check   :', 'NEGATIVE (claim)' if R['slope']<0 else 'POSITIVE -> wrong sign -> None')

slope        : +0.0177   NW(6) t = +6.39   R2 = 0.0127
correlation  : +0.1125   (n=329)
spread mean  : +0.251%/mo (sd 4.37%)
sign check   : POSITIVE -> wrong sign -> None


## It is one outlier — the 2020 claims spike

In [3]:
print(f"full sample   : slope {R['slope']:+.4f}  NW t = {R['t_nw']:+.2f}")
print(f"ex-COVID 2020 : slope {R['ex_covid_slope']:+.4f}  NW t = {R['ex_covid_t']:+.2f}  (n={R['ex_covid_n']})")
print(f"winsor 1/99   : slope {R['winsor_slope']:+.4f}  NW t = {R['winsor_t']:+.2f}")
print(f"Spearman rank : rho {R['spearman_rho']:+.4f}  p = {R['spearman_p']:.3f}")

full sample   : slope +0.0177  NW t = +6.39
ex-COVID 2020 : slope +0.0623  NW t = +1.49  (n=318)
winsor 1/99   : slope +0.0504  NW t = +1.43
Spearman rank : rho +0.0189  p = 0.733


## Robustness — two eras (split 2012-01-01)

In [4]:
print(f"1999-2011 (n={R['era_early_n']}): slope {R['era_early_slope']:+.4f}  NW t = {R['era_early_t']:+.2f}")
print(f"2012-2026 (n={R['era_late_n']}): slope {R['era_late_slope']:+.4f}  NW t = {R['era_late_t']:+.2f}  (<- contains 2020)")

1999-2011 (n=156): slope +0.1222  NW t = +1.41
2012-2026 (n=173): slope +0.0164  NW t = +6.83  (<- contains 2020)


## Placebo — shuffle the claims change vs the forward spread (2,000 draws)

In [5]:
print(f"observed {R['placebo_obs']:+.4f} vs placebo mean {R['placebo_mean']:+.5f} "
      f"(sd {R['placebo_sd']:.4f}) -> two-sided p = {R['placebo_p']:.3f}")

observed +0.0177 vs placebo mean -0.00003 (sd 0.0089) -> two-sided p = 0.053


## The timer — can you get paid for the rotation?

Flip a long-short cyclical/defensive book with the claim's sign; one-way × NAV per leg + 50 bps/yr borrow on the short.

In [6]:
for tag,g,n,t in [('0 bp',R['timer0_gross'],R['timer0_net'],R['timer0_t']),
                  ('10 bp',R['timer10_gross'],R['timer10_net'],R['timer10_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f}%/yr -> net {n:+.2f}%/yr (t_net {t:+.2f})")
print(f"({R['n_switches']} switches over the sample)")

 0 bp one-way: gross +0.89%/yr -> net +0.39%/yr (t_net +0.14)
10 bp one-way: gross +0.89%/yr -> net -1.91%/yr (t_net -0.66)
(168 switches over the sample)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted **negative** slope.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from claims_nowcast import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_frame(edge=0.0, seed=881+s, n_months=360))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_frame(edge=0.5, seed=881, n_months=360))
print(f"planted (edge=0.5): slope {planted['slope']:+.4f}, NW t = {planted['t_nw']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.09 (sd 1.13), |t|>=2 in 1/8
planted (edge=0.5): slope -0.4651, NW t = -13.85


## Verdict

- **Signal — None.** The claimed labour-nowcast rotation does not exist. The predictive slope is **wrong-signed** (positive, +6.39 *t*), and even that is a **single-outlier (COVID-2020)** artefact: *t* ≈ 1.5 ex-COVID / 1.4 winsorised, Spearman ρ = +0.02 (p = 0.73), pre-2020 era *t* = +1.41, placebo p = 0.05. The 20-seed synthetic control recovers a *planted* rotation (*t* = -13.85, fires on 1/20 nulls), so the engine is sound — the effect is absent.
- **Tradability — Mirage.** The sign-correct rotation earns only **+0.89%/yr gross** (*t* ≈ 0) and goes negative (**-1.91%/yr**) once its 168 flips pay a 10 bp one-way cost.